In [1]:
!pip install spacy scikit-learn nltk pandas matplotlib seaborn wordcloud plotly
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 46.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys

project_path = '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM'

sys.path.append(project_path)

In [4]:
import pandas as pd

df = pd.read_csv(f'{project_path}/data/BBC News Train.csv')
df.head()

,ArticleId,Text,Category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business


In [5]:
from newsbot.preprocessing import clean_text, preprocess_text
from newsbot.extract_syntactic_features import extract_syntactic_features
from newsbot.pos import analyze_pos_patterns
from newsbot.sentiment_analysis import analyze_sentiment

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [6]:
import joblib

label_encoder = joblib.load(f'{project_path}/newsbot/label_encoder.pkl')

tfidf_vectorizer = joblib.load(f'{project_path}/newsbot/tfidf_vectorizer.pkl')

model = joblib.load(f'{project_path}/newsbot/svm_model.pkl')

feature_columns = joblib.load(f'{project_path}/newsbot/feature_columns.pkl')

In [7]:
def NewsBotAgent(text):

    cleaned_text = clean_text(text)
    preprocessed_text = preprocess_text(cleaned_text)

    tfidf_matrix = tfidf_vectorizer.transform([preprocessed_text])
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

    pos_features = analyze_pos_patterns(cleaned_text)
    pos_df = pd.DataFrame([pos_features])

    combined_df = pd.concat([tfidf_df, pos_df], axis=1)
    final_text = combined_df.reindex(columns=feature_columns, fill_value=0)

    prediction = model.predict(final_text)
    category = label_encoder.inverse_transform(prediction)[0]

    sentiment = analyze_sentiment(text)
    syntactic_features = extract_syntactic_features(text)

    print(f"📊 Article's Category: {category}")
    print(f"🎭 The mood of the Article is: {sentiment}")
    print(f"🔍 Syntactic Features: {syntactic_features}")



    return {
          "category": category,
          "sentiment": sentiment,
          "syntactic_features": syntactic_features
      }

In [8]:
df_test = pd.read_csv(f'{project_path}/data/BBC News Test.csv')
df_test.head()

,ArticleId,Text
0,1018,qpr keeper day heads for preston queens park r...
1,1319,software watching while you work software that...
2,1138,d arcy injury adds to ireland woe gordon d arc...
3,459,india s reliance family feud heats up the ongo...
4,1020,boro suffer morrison injury blow middlesbrough...


In [9]:
sample_text = df_test['Text'][0]

NewsBotAgent(sample_text)

📊 Article's Category: sport
🎭 The mood of the Article is: {'neg': 0.031, 'neu': 0.873, 'pos': 0.097, 'compound': 0.8735, 'sentiment_label': 'positive'}
🔍 Syntactic Features: Sentences: 10, Subjects: ['heads', 'day', 'who', 'qpr', 'holloway', 'some', 'it', 'he', 't', 'royce', 'i', 'i', 'i', 'contract', 'holloway', 'davies', 'holloway'], Objects: ['day', 'preston', 'loan', 'arrival', 'royce', 'month', 'loan', 'charlton', 'rossi', 'month', 'charlton', 'irons', 'fire', 'yes', 'couple', 'others', 'them', 'summer', 'signing', 'davies', 'loan', 'match', 'ipswich', 'spell', 'road', 'doherty'], Noun phrases: ['qpr keeper day heads', 'preston queens park rangers keeper chris day', 'preston', 'a month s loan', 'day', 'the arrival', 'simon royce', 'who', 'his second month', 'loan', 'charlton', 'qpr', 'italian generoso rossi', 'r s manager ian holloway', 'some', 'it', 'a risk', 'he', 't', 'that month', 'simon royce', 'charlton', 'i', 'other irons', 'the fire', 'i', 'a  yes', 'a couple', 'others', '

{'category': 'sport',
 'sentiment': {'neg': 0.031,
  'neu': 0.873,
  'pos': 0.097,
  'compound': 0.8735,
  'sentiment_label': 'positive'},
 'syntactic_features': "Sentences: 10, Subjects: ['heads', 'day', 'who', 'qpr', 'holloway', 'some', 'it', 'he', 't', 'royce', 'i', 'i', 'i', 'contract', 'holloway', 'davies', 'holloway'], Objects: ['day', 'preston', 'loan', 'arrival', 'royce', 'month', 'loan', 'charlton', 'rossi', 'month', 'charlton', 'irons', 'fire', 'yes', 'couple', 'others', 'them', 'summer', 'signing', 'davies', 'loan', 'match', 'ipswich', 'spell', 'road', 'doherty'], Noun phrases: ['qpr keeper day heads', 'preston queens park rangers keeper chris day', 'preston', 'a month s loan', 'day', 'the arrival', 'simon royce', 'who', 'his second month', 'loan', 'charlton', 'qpr', 'italian generoso rossi', 'r s manager ian holloway', 'some', 'it', 'a risk', 'he', 't', 'that month', 'simon royce', 'charlton', 'i', 'other irons', 'the fire', 'i', 'a  yes', 'a couple', 'others', 'i', 'them',

In [10]:
venezuela_article = "More than 900 people have been killed and 3,360 others injured in the Venezuela earthquakes, according to the government, as rescuers keep searching for survivors and families wait desperately for news. The injured are being treated in makeshift medical facilities after dozens of buildings in the country's north were destroyed by the two quakes, including in the capital Caracas. UN humanitarian chief Tom Fletcher says almost 2,000 international rescue workers are part of the response. Two powerful earthquakes rocked Venezuela within seconds of each other on Wednesday. The second quake was one of the strongest tremors to hit the country in a century, at a magnitude of 7.5."

spain_article = "Spain defeated Uruguay 1-0 after another goalkeeping mistake by Fernando Muslera, advancing to the knockout stage of the World Cup and eliminating the South American powerhouse on Friday. Uruguay, a two-time champion, will go home without any victories in its three Group H games. Spain, the European champion, won the group with seven points and will face the second-place team from Group J — either Austria or Algeria — on Thursday in Inglewood, California. Álex Baena scored in the 42nd minute after Muslera couldn’t fully swat away his shot from inside the area. It was the third blunder of the tournament by the 40-year-old Muslera, who asked coach Marcelo Bielsa to substitute him at halftime. Sergio Rochet came in to start the second half."

disney_article = "The Walt Disney empire is to buy the superheroes stable Marvel Entertainment for $4bn (£2.5bn) in a star-studded Hollywood deal that unites family names such as Mickey Mouse with lucrative characters including Spider-Man, the Incredible Hulk and the X-Men."

NewsBotAgent(venezuela_article)
NewsBotAgent(spain_article)
NewsBotAgent(disney_article)

📊 Article's Category: business
🎭 The mood of the Article is: {'neg': 0.128, 'neu': 0.8, 'pos': 0.071, 'compound': -0.7884, 'sentiment_label': 'negative'}
🔍 Syntactic Features: Sentences: 5, Subjects: ['people', 'rescuers', 'injured', 'dozens', 'fletcher', 'workers', 'earthquakes', 'quake'], Objects: ['earthquakes', 'government', 'survivors', 'news', 'facilities', 'buildings', 'north', 'quakes', 'capital', 'response', 'venezuela', 'seconds', 'other', 'wednesday', 'tremors', 'country', 'century', 'magnitude', '7.5'], Noun phrases: ['more than 900 people', '3,360 others', 'the venezuela earthquakes', 'the government', 'rescuers', 'survivors', 'families', 'news', 'makeshift medical facilities', 'dozens', 'buildings', "the country's north", 'the two quakes', 'the capital', 'caracas', 'un humanitarian chief tom fletcher', 'almost 2,000 international rescue workers', 'part', 'the response', 'two powerful earthquakes', 'venezuela', 'seconds', 'wednesday', 'the second quake', 'the strongest tre

{'category': 'business',
 'sentiment': {'neg': 0.0,
  'neu': 0.818,
  'pos': 0.182,
  'compound': 0.7783,
  'sentiment_label': 'positive'},
 'syntactic_features': "Sentences: 1, Subjects: ['empire', 'that'], Objects: ['superheroes', 'entertainment', '4bn', 'deal', 'names', 'mouse', 'characters', 'man'], Noun phrases: ['the walt disney empire', 'the superheroes', 'marvel entertainment', '$4bn', 'a star-studded hollywood deal', 'that', 'family names', 'mickey mouse', 'lucrative characters', 'spider-man', 'the incredible hulk', 'the x', '-', 'men']"}